# Translator English to Spanish

In [4]:
from tqdm.notebook import tqdm
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from torchvision.transforms import ToTensor
import sys
from sklearn.model_selection import train_test_split
from dataset import tiktoken
from dataset import Dataset
from dataset import Tokenizer
from transformer import Transformer
import os

dataset_path = os.path.join("dataset","dataset.json")

%reload_ext autoreload
%autoreload 2

## Dataset Análisis

In [5]:
import pandas as pd
import numpy as np
import json


with open(dataset_path, 'r', encoding='utf-8') as f:
    data = json.load(f)
    
    # Convertir a DataFrame
    df = pd.DataFrame(data)

    # 2. Calcular longitudes (Palabras)
    df['word_en'] = df['en'].apply(lambda x: len(str(x).split()))
    df['word_es'] = df['es'].apply(lambda x: len(str(x).split()))

    
    # (25%, 50%/Mediana, 75%, 90% y 95%)
    percentiles = [.25, .5, .75, .9, .95]

    # 4. Generar estadísticas descriptivas
    # Usamos .describe() para obtener media, desv. estándar, min, max y percentiles
    estadisticas = df[[ 'word_en', 'word_es']].describe(percentiles=percentiles)

pd.options.display.float_format = '{:.2f}'.format
estadisticas

,word_en,word_es
count,600000.00,600000.00
mean,22.73,23.80
std,15.45,16.64
min,0.00,0.00
25%,11.00,11.00
50%,20.00,21.00
75%,31.00,32.00
90%,43.00,45.00
95%,51.00,54.00
max,668.00,658.00


## Vocabulario y Tokens


Una vez tenemos el dataset, necesitamos tokenizar la entrada pues es lo que el modelo acepta como entrada.


Inicialmente, se pensó usar el vocabulario de GPT2 con 52.000 tokens, pero dicho vocabulario se entrena solo con frases en inglés y hace que al modelo le cueste aprender.

Lo mejor es crear nuestro propio vocabulario mediante el algoritmo Byte Pair Encoding.

Crearemos un vocabulario formado por 15.000 tokens

In [6]:
import json
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

vocabulario = "vocabulario.json"

# 2. Función generadora para pasarle el texto al entrenador sin saturar la RAM
def generador_de_textos(ruta_json):
    with open(ruta_json, 'r', encoding='utf-8') as f:
        datos = json.load(f)
        for par in datos:
            # Mandamos a entrenar tanto la frase en inglés como en español
            yield par['en']
            yield par['es']

# 3. Inicializamos el Tokenizador BPE
# Todo lo que no conozca lo marcará como <UNK> 
tokenizer = Tokenizer(BPE(unk_token="<UNK>"))

tokenizer.pre_tokenizer = Whitespace() # separar por espacios en blanco antes de aplicar el BPE


# 15,000 tamaño vocab 
entrenador = BpeTrainer(
    vocab_size=15000, 
    special_tokens=["<PAD>", "<START>", "<END>", "<UNK>"]
)


print("Entrenando el tokenizador...")
# 5. ¡A entrenar! Le pasamos el generador
tokenizer.train_from_iterator(generador_de_textos(dataset_path), trainer=entrenador)

# 6. Guardamos el tokenizador entrenado en un archivo para usarlo siempre
tokenizer.save(vocabulario)

print(f"Tamaño final del vocabulario: {tokenizer.get_vocab_size()} tokens.")

Entrenando el tokenizador...
Tamaño final del vocabulario: 15000 tokens.


## Pruebas

Una vez entrenado el modelo, realizamos pruebas y vemos el comportamiento del modelo

In [ ]:
dataset_path = "dataset.json"
vocab_path = "vocabulario.json"

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
tokenizer = Tokenizer(vocab_path)
transformer = Transformer(vocab_size=len(tokenizer),pad_id= tokenizer.pad_id()).to(device=device)

#Cargamos el modelo 
weights  = torch.load(os.path.join("resultados","6_val_2.6574.pth"), map_location=device, weights_only=True)
transformer.load_state_dict(weights,strict=False)

In [ ]:
# --- LISTA DE FRASES PARA TEST AMPLIADA ---
frases_test = [
    "I'm so happy to see you!",                       
    "Is Chae Yoon's coordinator in here?",           
    "What a wonderful surprise!",                    
    "Where are you going right now?",                
    "Stop talking and listen to me!",                
    "Do you want to eat kimchee with me?",            
    "I can't believe it's finally working!",          
    "How much does this book cost?",                  
    "Watch out! The floor is wet.",                  
    "Is it going to rain tomorrow?",                  
    "I love this song, it's my favorite!",            
    "Can you help me with my homework, please?"       
]


print(f"{'#':<3} | {'ENTRADA (EN)':<45} | {'TRADUCCIÓN (ES)':<45}")
print("-" * 100)

transformer.eval() # Aseguramos modo evaluación

for i, frase in enumerate(frases_test):
   

    x_tokens = torch.tensor(np.array([tokenizer.encode(frase, pad=False)])).to('cuda')
    y_tokens = torch.tensor(np.array([[tokenizer.star_of_text_id()]])).to('cuda')
    
    output = transformer.predict(
        x_tokens, 
        y_tokens,
        end_token_id=tokenizer.end_of_text_id(),
        device='cuda'
    )
    
    #output[0] porque es un batch de 1)
    traduccion = tokenizer.decoder(output[0])
    
    print(f"{i+1:<3} | {frase:<45} | {traduccion:<45}")
    if (i+1) % 5 == 0:
        print("-" * 100)


print("-" * 100)